# LoRA Fine-Tuning Lab: Six Parameter-Efficient Adaptation Methods

A Python-first lab covering **LoRA, QLoRA, AdaLoRA, rsLoRA, DoRA, and VeRA** with equations, PEFT configuration patterns, experiments, evaluation guidance, and MLOps considerations.

> Compute note: the mathematical experiments run on CPU. Real-model training is optional and may require a GPU, model access approval, and substantial VRAM. Check each model card, license, and current PEFT API before running.

## 1. LoRA fundamentals

A linear layer computes \(y=Wx+b\). Full fine-tuning updates \(W\), while LoRA freezes it and learns a low-rank update:

\[
W'=W+\Delta W,\qquad \Delta W=BA
\]

For \(W\in\mathbb{R}^{d_{out}\times d_{in}}\), choose rank \(r\ll\min(d_{in},d_{out})\), with \(A\in\mathbb{R}^{r\times d_{in}}\) and \(B\in\mathbb{R}^{d_{out}\times r}\).

The common LoRA forward path is:

\[
y=Wx + \frac{\alpha}{r}BAx+b
\]

The trainable parameter count changes from \(d_{out}d_{in}\) to approximately \(r(d_{in}+d_{out})\). Common targets in decoder models include attention projections such as `q_proj` and `v_proj`, but always inspect the selected model's module names.

In [ ]:
def parameter_counts(d_in: int, d_out: int, rank: int) -> dict:
    full = d_in * d_out
    adapter = rank * (d_in + d_out)
    return {
        "full_parameters": full,
        "adapter_parameters": adapter,
        "trainable_fraction": adapter / full,
        "reduction_factor": full / adapter,
    }

for r in (4, 8, 16, 32, 64, 128):
    print(r, parameter_counts(4096, 4096, r))

## 2. A tiny LoRA layer using NumPy

This illustrates tensor shapes and the low-rank update. It is a teaching example, not a production PEFT implementation.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
batch, d_in, d_out, rank = 8, 32, 24, 4
W = rng.normal(0, 0.02, (d_out, d_in)).astype(np.float32)
A = rng.normal(0, 0.02, (rank, d_in)).astype(np.float32)
B = np.zeros((d_out, rank), dtype=np.float32)
X = rng.normal(size=(batch, d_in)).astype(np.float32)
alpha = 8.0

base = X @ W.T
adapter = (X @ A.T) @ B.T
Y = base + (alpha / rank) * adapter

print("X", X.shape, "W", W.shape, "A", A.shape, "B", B.shape, "Y", Y.shape)
print("Initial adapter contribution:", np.linalg.norm(adapter))

`B` starts at zero here, so the adapter initially contributes zero and the layer begins equivalent to the frozen base layer.

## 3. Six methods compared

| Method | Core idea | Why use it | Main trade-off |
|---|---|---|---|
| **LoRA** | Learn low-rank matrices A and B | Strong, simple baseline | Fixed rank; conventional scaling |
| **QLoRA** | Quantize frozen base weights and train LoRA | Lower GPU memory | Quantization/kernel compatibility and possible quality changes |
| **AdaLoRA** | Adapt rank allocation across matrices | Allocate capacity selectively | More schedule and tuning complexity |
| **rsLoRA** | Use rank-stabilized scaling | Better scaling behavior at higher ranks | Must record and tune scaling mode |
| **DoRA** | Decompose weights into magnitude and direction | More expressive adaptation in some tasks | More compute and implementation complexity |
| **VeRA** | Share frozen random low-rank matrices; learn scaling vectors | Very small trainable footprint | Different capacity/quality trade-offs |

These are parameterization and training strategies, not six separate foundation-model architectures.

## 4. Method 1 — Standard LoRA

Standard LoRA freezes base weights and learns A and B. The common scaling factor is \(s=\alpha/r\). Rank controls adapter capacity; alpha controls update scale. Tune them together.

Typical uses: domain adaptation, instruction tuning, output-format adaptation, and task-specific generation when the base model already has the required general capabilities.

In [ ]:
# Optional: pip install -U transformers peft accelerate torch
from peft import LoraConfig, TaskType

standard_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],  # inspect model first
    bias="none",
)
print(standard_config)

**Experiment:** compare ranks 4, 8, 16, and 32 while holding the dataset, seed, training tokens, optimizer budget, and evaluation set fixed.

## 5. Method 2 — QLoRA

QLoRA combines a quantized frozen base model with trainable LoRA adapters. A common recipe uses 4-bit NF4 weights, a higher-precision compute dtype, and memory-saving optimizer techniques where supported.

- Quantization compresses the frozen base weights.
- LoRA supplies the trainable low-rank update.
- The base model is generally not fully fine-tuned in higher precision.

QLoRA is not merely LoRA with a smaller rank; its principal memory savings come from base-model quantization.

In [ ]:
from transformers import BitsAndBytesConfig
import torch

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
print(quant_config)

QLoRA workflow: load the quantized base model; prepare it for k-bit training with PEFT; attach LoRA; train adapter parameters; save adapter and config; evaluate using the intended quantization and serving path.

Trade-offs: throughput depends on GPU architecture, kernels, sequence length, batch size, and attention implementation. Quantization may affect quality, so validate task metrics and generation behavior.

## 6. Method 3 — AdaLoRA (Adaptive LoRA)

Standard LoRA often assigns the same rank to every target matrix. AdaLoRA adapts rank allocation during training, aiming to spend more rank budget on important updates and less on less-important ones.

This can help when layers or projection matrices contribute unequally to a task. It adds schedule complexity, and short training runs may not provide enough signal for useful allocation.

In [ ]:
from peft import AdaLoraConfig, TaskType

adalora_config = AdaLoraConfig(
    task_type=TaskType.CAUSAL_LM,
    init_r=12,
    target_r=4,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    total_step=1000,  # planned optimizer steps
    tinit=100,
    tfinal=100,
    deltaT=10,
    beta1=0.85,
    beta2=0.85,
)
print(adalora_config)

These values are illustrative. Set `total_step`, warm-up/finalization periods, and update intervals to match the real training schedule and the PEFT release you pin.

## 7. Method 4 — rsLoRA (Rank-Stabilized LoRA)

Standard LoRA commonly scales by \(\alpha/r\). rsLoRA uses:

\[
s_{rs}=\frac{\alpha}{\sqrt{r}}
\]

The motivation is to stabilize learning behavior as rank grows, rather than allowing the conventional scaling factor to shrink proportionally to rank.

In [ ]:
from peft import LoraConfig, TaskType

rslora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=64,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    use_rslora=True,
)
print(rslora_config)

alpha = 32
for rank in (4, 8, 16, 32, 64, 128):
    print(rank, "standard:", alpha/rank, "rsLoRA:", alpha/(rank**0.5))

Use rsLoRA when exploring higher ranks and rank sweeps. Do not assume standard LoRA and rsLoRA have equivalent update scales when alpha is unchanged. Record the scaling mode in experiment metadata.

## 8. Method 5 — DoRA (Weight-Decomposed Low-Rank Adaptation)

DoRA decomposes a weight into magnitude and direction, then adapts these components. A simplified conceptual view is:

\[
W=m\cdot\frac{V}{\|V\|}
\]

Here \(m\) represents magnitude and \(V\) direction. DoRA applies low-rank adaptation to the directional component while learning magnitude-related parameters. Exact tensor operations are implementation-specific.

In [ ]:
from peft import LoraConfig, TaskType

dora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    use_dora=True,
)
print(dora_config)

DoRA can improve adaptation quality over standard LoRA in some settings. It adds operations and may behave differently across training and inference backends. Benchmark the actual serving path rather than assuming standard-LoRA latency.

## 9. Method 6 — VeRA (Vector-based Random Matrix Adaptation)

VeRA reduces trainable parameters by using shared frozen random low-rank matrices and learning small scaling vectors. A simplified form is:

\[
\Delta W_i=\Lambda_{b,i}B\Lambda_{d,i}A
\]

The random projection matrices A and B are shared and frozen; learned vectors scale the projections for each layer. This can reduce adapter storage substantially.

In [ ]:
from peft import VeraConfig, TaskType

vera_config = VeraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=256,
    target_modules=["q_proj", "v_proj"],
    vera_dropout=0.05,
    d_initial=0.1,
    projection_prng_key=42,
)
print(vera_config)

VeRA is attractive when storing many task-specific adapters or minimizing trainable parameters. Its capacity differs from conventional LoRA, so measure quality rather than assuming fewer parameters preserve the same result.

> PEFT APIs evolve. If your installed release does not expose `VeraConfig` or one of these fields, consult that version's documentation and pin a tested release.

## 10. Load a real model and inspect its modules (optional)

Use a small accessible checkpoint for the first run. Substitute a Mistral-family checkpoint only after checking its model card, license, access requirements, memory needs, and target module names.

In [ ]:
# Uncomment after selecting a model you are authorized to use.
#
# from transformers import AutoModelForCausalLM, AutoTokenizer
# model_id = "Qwen/Qwen2.5-0.5B-Instruct"  # example; verify current model card
# tokenizer = AutoTokenizer.from_pretrained(model_id)
# model = AutoModelForCausalLM.from_pretrained(model_id)
#
# for name, module in model.named_modules():
#     if any(key in name for key in ("q_proj", "k_proj", "v_proj", "o_proj")):
#         print(name, type(module).__name__)

### Attach an adapter

Only run after loading `model` and confirming that the configured target modules exist.

In [ ]:
# from peft import get_peft_model
# model = get_peft_model(model, standard_config)
# model.print_trainable_parameters()

## 11. Small supervised fine-tuning data example

The training record format must match the model's chat template. A generic prompt format is not necessarily equivalent to the model's official template.

In [ ]:
records = [
    {
        "instruction": "Explain approximate nearest-neighbor search.",
        "context": "The corpus contains several million embedded text chunks.",
        "response": "ANN indexes trade a small amount of exactness for lower search latency at scale.",
    },
    {
        "instruction": "Summarize the incident.",
        "context": "The inference endpoint exceeded its p95 latency SLO for 8 minutes.",
        "response": "The endpoint breached its p95 latency SLO for 8 minutes; investigate traffic, queues, model latency, and dependencies.",
    },
]

def format_record(row: dict) -> str:
    required = {"instruction", "context", "response"}
    missing = required - row.keys()
    if missing:
        raise ValueError(f"Missing fields: {sorted(missing)}")
    return (
        f"### Instruction\n{row['instruction'].strip()}\n\n"
        f"### Context\n{row['context'].strip()}\n\n"
        f"### Response\n{row['response'].strip()}"
    )

for row in records:
    print(format_record(row), "\n" + "-"*50)

## 12. Training workflow scaffold

A typical Hugging Face workflow:

1. Tokenize examples and prepare labels.
2. Load the pretrained model and tokenizer.
3. Freeze base weights and attach the selected adapter.
4. Train with task-appropriate loss (causal LM commonly uses next-token cross-entropy).
5. Evaluate on held-out examples.
6. Save adapter weights and config.
7. Reload base model + adapter for inference.

This scaffold is intentionally commented: dataset, collator, model, and training arguments must be chosen for the task and compute environment.

In [ ]:
# from transformers import Trainer, TrainingArguments
#
# args = TrainingArguments(
#     output_dir="./adapter-output",
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=8,
#     learning_rate=2e-4,
#     num_train_epochs=1,
#     logging_steps=10,
#     save_strategy="epoch",
#     report_to="none",
# )
#
# trainer = Trainer(
#     model=model,
#     args=args,
#     train_dataset=tokenized_train,
#     eval_dataset=tokenized_eval,
#     data_collator=data_collator,
# )
# trainer.train()
# model.save_pretrained("./adapter-output/final")
# tokenizer.save_pretrained("./adapter-output/final")

## 13. Evaluation: quality, cost, and operations

Do not select an adapter from training loss alone. Record:

- Task quality: exact match, F1, ROUGE, pass rate, or domain metric.
- Generalization on held-out and out-of-domain examples.
- Trainable parameter count and adapter artifact size.
- Training time, peak GPU memory, tokens/second.
- Inference p50/p95 latency, throughput, and memory.
- Robustness to long contexts, malformed inputs, and prompt variation.

Control the base model revision, dataset split, chat template, sequence length, training-token budget, optimizer budget, evaluation prompts, and serving hardware.

In [ ]:
from dataclasses import dataclass, asdict
import json

@dataclass
class ExperimentResult:
    method: str
    rank: int | None
    trainable_parameters: int | None = None
    adapter_size_mb: float | None = None
    train_minutes: float | None = None
    peak_gpu_memory_gb: float | None = None
    eval_score: float | None = None
    p95_latency_ms: float | None = None

results = [
    ExperimentResult("LoRA", 16),
    ExperimentResult("QLoRA", 16),
    ExperimentResult("AdaLoRA", 4),
    ExperimentResult("rsLoRA", 64),
    ExperimentResult("DoRA", 16),
    ExperimentResult("VeRA", 256),
]
print(json.dumps([asdict(x) for x in results], indent=2))

## 14. Suggested controlled experiment matrix

| Run | Method | Starting configuration | Main question |
|---|---|---|---|
| 1 | LoRA | r=16, alpha=32 | Baseline quality and cost? |
| 2 | QLoRA | r=16, 4-bit NF4 | How much memory is saved? |
| 3 | AdaLoRA | init_r=12, target_r=4 | Does adaptive allocation help? |
| 4 | rsLoRA | r=64, alpha=32 | Does rank-stabilized scaling help at higher rank? |
| 5 | DoRA | r=16, alpha=32 | Does weight decomposition improve quality? |
| 6 | VeRA | r=256 | What is the quality/storage trade-off? |

These are starting points, not guaranteed optimal values. Repeat promising runs with multiple seeds and compare at similar token, parameter, or compute budgets.

## 15. MLOps and deployment considerations

Treat each adapter as a versioned model artifact with lineage. Capture:

- Base model ID and immutable revision.
- Adapter method and complete configuration.
- Dataset version, preprocessing version, and split IDs.
- Tokenizer and chat-template revision.
- Training code commit, dependency lockfile, random seed.
- Hardware, precision, quantization, optimizer settings.
- Evaluation metrics, test-set version, artifact checksum, approval status.

Serving patterns:
1. Load base model plus adapter at runtime to switch tasks; this adds adapter-management complexity.
2. Merge compatible adapters into base weights where supported; validate precision and quantization behavior.
3. Run separate endpoints per adapter; this isolates deployments but may increase GPU/model-loading costs.

For Mistral-based serving, test tokenizer/template consistency, context limits, concurrency, batching, KV-cache memory, and p95/p99 latency.

## 16. Knowledge check

1. Which method quantizes the frozen base model? **QLoRA**
2. Which method commonly uses alpha divided by square-root rank? **rsLoRA**
3. Which method adaptively allocates rank? **AdaLoRA**
4. Which method decomposes magnitude and direction? **DoRA**
5. Which method uses shared frozen random matrices and learned scaling vectors? **VeRA**

## 17. Recommended next steps

1. Run the NumPy parameter-count and matrix examples.
2. Pin compatible versions of `torch`, `transformers`, `accelerate`, `peft`, and (for QLoRA) `bitsandbytes`.
3. Select a small accessible model and inspect its projection module names.
4. Train and evaluate standard LoRA first.
5. Add one method at a time with controlled comparisons.
6. Export experiment metadata and metrics for reproducibility.

**Version note:** PEFT and Transformers APIs change. Pin tested versions and verify the exact configuration fields exposed by your installed release before starting a long training run.